# 10 — Build the Multi-Agent Orchestrator

We now have two specialized capabilities:

```text
Research Agent
    ↓
Unstructured research + AI Search

Portfolio Agent
    ↓
Structured analytics + Genie + Metric View
```

This notebook introduces the **Multi-Agent Orchestrator**.

The orchestrator receives one user question, decides which specialist is needed, calls the appropriate agent, and combines the results into one answer.

### Target flow

```text
                         User Question
                              ↓
                    Multi-Agent Orchestrator
                              ↓
                 Claude Sonnet through Unity Gateway
                              ↓
            ┌─────────────────┴─────────────────┐
            │                                   │
     Research Agent                      Portfolio Agent
            │                                   │
       AI Search                         Genie Agent
            │                                   │
  Research evidence                    portfolio_metrics
            └─────────────────┬─────────────────┘
                              ↓
                    One coordinated answer
```

This notebook is our **transparent orchestration prototype**.

For production deployment, we will package the orchestrator using the current Databricks **Agent Bricks** path rather than the legacy Supervisor Agent pattern.

## 1. Install the required libraries

The orchestrator needs the same building blocks we already used:

- Anthropic SDK for Claude and tool calling,
- Databricks AI Bridge retrieval for the Research Agent,
- Databricks AI Bridge Genie client for the Portfolio Agent,
- MLflow for tracing.

In [0]:
%pip install -q --upgrade databricks-openai databricks-ai-bridge anthropic "mlflow[databricks]>=3.5"
dbutils.library.restartPython()

## 2. Configure MLflow tracing

We use one dedicated experiment for the orchestrator.

This lets us inspect the parent orchestration call together with the Claude reasoning and tool calls that happen underneath it.

In [0]:
import mlflow

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/investment_ai_multi_agent_orchestrator")
mlflow.anthropic.autolog()

print("✅ MLflow tracing configured.")
print("Experiment: /Shared/investment_ai_multi_agent_orchestrator")

## 3. Configure the shared Databricks and Claude connection

The Research Agent and the Orchestrator both use our governed Anthropic Model Provider Service through Unity Gateway.

The Anthropic API key remains behind Unity Gateway.  
The notebook authenticates with Databricks credentials only.

In [0]:
import json

from anthropic import AsyncAnthropic
from databricks.sdk import WorkspaceClient
from databricks_openai import VectorSearchRetrieverTool
from databricks_ai_bridge.genie import Genie

MODEL_PROVIDER_SERVICE = "investment_ai.gateway.anthropic_claude"
ORCHESTRATOR_MODEL = "claude-sonnet-5-5"
RESEARCH_MODEL = "claude-sonnet-5-5"

AI_SEARCH_INDEX = "investment_ai.research.investment_ai_research_index"

GENIE_AGENT_ID = "01f1cb3f9bdd1714ae1bb5ad29f10f32"
GENIE_AGENT_NAME = "Investment AI Portfolio Agent"

workspace = WorkspaceClient()

auth_header = workspace.config.authenticate()["Authorization"]
databricks_token = auth_header.replace("Bearer ", "", 1)

UNITY_GATEWAY_ANTHROPIC_URL = (
    f"{workspace.config.host.rstrip('/')}/ai-gateway/anthropic"
)

anthropic_client = AsyncAnthropic(
    auth_token=databricks_token,
    base_url=UNITY_GATEWAY_ANTHROPIC_URL,
    default_headers={
        "Databricks-Model-Provider-Service": MODEL_PROVIDER_SERVICE,
    },
)

genie = Genie(
    space_id=GENIE_AGENT_ID,
    return_pandas=False,
)

print("✅ Shared agent connections configured.")
print("Orchestrator model:", ORCHESTRATOR_MODEL)
print("Research index    :", AI_SEARCH_INDEX)
print("Portfolio agent   :", GENIE_AGENT_NAME)

## 4. Recreate the Research Agent capability

The orchestrator should not query research documents itself.

It delegates unstructured investment-research work to the specialized Research Agent we built earlier.

The Research Agent still owns:

- earnings and guidance,
- management commentary,
- reasons for business changes,
- risks and drivers,
- evidence and citations.

In [0]:
RETURN_COLUMNS = [
    "chunk_to_retrieve",
    "security_id",
    "company_name",
    "document_type",
    "source_type",
    "source_authority_rank",
    "publish_date",
    "citation_label",
]

def _run_research_search(
    query: str,
    security_id: str | None = None,
    fiscal_quarter: str | None = None,
    fiscal_year: int | None = None,
    source_type: str | None = None,
):
    filters = {}

    if security_id:
        filters["security_id"] = security_id
    if fiscal_quarter:
        filters["fiscal_quarter"] = fiscal_quarter
    if fiscal_year:
        filters["fiscal_year"] = fiscal_year
    if source_type:
        filters["source_type"] = source_type

    retriever = VectorSearchRetrieverTool(
        index_name=AI_SEARCH_INDEX,
        tool_name="investment_research_search",
        tool_description=(
            "Search the governed Investment AI research corpus for earnings, "
            "guidance, outlook, management commentary, business drivers, and risks."
        ),
        num_results=5,
        columns=RETURN_COLUMNS,
        filters=filters or None,
        query_type="HYBRID",
    )

    return retriever.execute(query=query)


def search_investment_research(
    query: str,
    security_id: str | None = None,
    fiscal_quarter: str | None = None,
    fiscal_year: int | None = None,
    source_type: str | None = None,
) -> str:
    results = _run_research_search(
        query=query,
        security_id=security_id,
        fiscal_quarter=fiscal_quarter,
        fiscal_year=fiscal_year,
        source_type=source_type,
    )
    return json.dumps(results, default=str)


SEARCH_TOOL_DEFINITION = {
    "name": "search_investment_research",
    "description": (
        "Search the governed synthetic investment research corpus. "
        "Use for company outlook, guidance, earnings commentary, "
        "business drivers, risks, management statements, and research evidence."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "security_id": {"type": "string"},
            "fiscal_quarter": {"type": "string"},
            "fiscal_year": {"type": "integer"},
            "source_type": {"type": "string"},
        },
        "required": ["query"],
    },
}

RESEARCH_AGENT_INSTRUCTIONS = '''
You are the Investment AI Research Agent.

Your responsibility is unstructured investment research only.

Use search_investment_research for factual questions about:
- company guidance or outlook,
- earnings commentary,
- reasons for business changes,
- company risks or drivers,
- management statements,
- research evidence.

Evidence rules:
- Ground factual research claims in retrieved evidence.
- Prefer primary sources over internal sources, and internal over secondary sources
  when evidence conflicts.
- Use citation_label from retrieved results.
- Never invent citations or facts.
- If evidence is incomplete or conflicting, say so.

Scope boundary:
- You do not have portfolio holdings, ownership, exposure, or fund-position data.
- Portfolio questions belong to the Portfolio Agent.

Keep answers concise, analytical, and evidence-based.
'''


async def run_research_agent(
    user_question: str,
    max_iterations: int = 10,
) -> str:
    messages = [{"role": "user", "content": user_question}]

    for _ in range(max_iterations):
        response = await anthropic_client.messages.create(
            model=RESEARCH_MODEL,
            max_tokens=4096,
            system=RESEARCH_AGENT_INSTRUCTIONS,
            messages=messages,
            tools=[SEARCH_TOOL_DEFINITION],
        )

        text_parts = []
        tool_results = []
        has_tool_use = False

        for block in response.content:
            if block.type == "text":
                text_parts.append(block.text)

            elif block.type == "tool_use":
                has_tool_use = True
                result = search_investment_research(**block.input)

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })

        if has_tool_use:
            messages.append({
                "role": "assistant",
                "content": response.content,
            })
            messages.append({
                "role": "user",
                "content": tool_results,
            })
        else:
            return "\n".join(text_parts)

    return "Research Agent reached its maximum tool-call iterations."


print("✅ Research Agent capability ready.")

## 5. Recreate the Portfolio Agent capability

The Portfolio Agent remains our curated Genie Agent.

It owns structured questions about:

- holdings and ownership,
- shares or quantity,
- market value,
- cost basis,
- unrealized P&L,
- portfolio and sector exposure.

The orchestrator does not write portfolio SQL itself.  
It delegates that responsibility to Genie.

In [0]:
def ask_portfolio_agent(
    question: str,
    conversation_id: str | None = None,
) -> dict:
    response = genie.ask_question(
        question,
        conversation_id=conversation_id,
    )

    return {
        "answer": response.result,
        "sql": getattr(response, "query", None),
        "conversation_id": response.conversation_id,
        "suggested_questions": getattr(
            response,
            "suggested_questions",
            None,
        ),
    }


print("✅ Portfolio Agent capability ready.")

## 6. Expose both specialized agents as orchestrator tools

This is the key multi-agent transition.

The orchestrator does not receive direct access to AI Search or the portfolio tables.

Instead, it receives **two specialist-agent tools**:

```text
ask_research_agent
ask_portfolio_agent
```

The tool descriptions tell Claude what each specialist is responsible for.

In [0]:
ORCHESTRATOR_TOOLS = [
    {
        "name": "ask_research_agent",
        "description": (
            "Delegate unstructured investment-research questions to the specialized "
            "Research Agent. Use for earnings, guidance, outlook changes, management "
            "commentary, business drivers, risks, and research evidence."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "A complete research question for the Research Agent.",
                }
            },
            "required": ["question"],
        },
    },
    {
        "name": "ask_portfolio_agent",
        "description": (
            "Delegate structured portfolio questions to the curated Genie Portfolio "
            "Agent. Use for holdings, ownership, quantity, market value, cost basis, "
            "unrealized P&L, portfolio exposure, sector exposure, and position analytics."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "A complete portfolio analytics question for the Portfolio Agent.",
                }
            },
            "required": ["question"],
        },
    },
]

print("✅ Specialized agents exposed as orchestrator tools.")

## 7. Define the orchestrator behavior

The orchestrator's job is **coordination**, not domain expertise.

It should:

1. understand the user's overall request,
2. decide which specialist agent is needed,
3. call one or both specialists,
4. use the returned results,
5. synthesize one final answer.

For mixed research + portfolio questions, the Research Agent may need to run first so the orchestrator can discover the relevant companies or security IDs before asking the Portfolio Agent about ownership.

In [0]:
ORCHESTRATOR_INSTRUCTIONS = '''
You are the Investment AI Multi-Agent Orchestrator.

Your responsibility is to coordinate specialized agents and produce one coherent answer.

Available specialists:

1. Research Agent
   Use for:
   - earnings and guidance,
   - changes in company outlook,
   - management commentary,
   - business drivers and risks,
   - research evidence and citations.

2. Portfolio Agent
   Use for:
   - holdings and ownership,
   - shares or quantity,
   - market value,
   - cost basis,
   - unrealized gain/loss,
   - company, portfolio, or sector exposure.

Routing rules:

- For research-only questions, call only ask_research_agent.
- For portfolio-only questions, call only ask_portfolio_agent.
- For questions that need both research and portfolio information, call both.
- When a portfolio question depends on companies discovered from research, use the
  Research Agent result first, then send the relevant company names or security IDs
  to the Portfolio Agent.
- Do not bypass a specialist by inventing domain facts yourself.
- Preserve research citations returned by the Research Agent.
- When reporting portfolio information, make the portfolio snapshot date clear.
- If a specialist says information is unavailable, state that clearly rather than
  inventing an answer.

For combined answers, clearly connect the research finding with the corresponding
portfolio exposure so the result reads like one investment-analysis response,
not two unrelated outputs.

Keep the final answer concise, analytical, and easy to read.
'''

## 8. Build the orchestration loop

This loop gives Claude control over **which specialist to call**.

A single user question can therefore produce:

```text
Research only
Portfolio only
or
Research → Portfolio → Combined answer
```

The notebook also records which specialist tools were used so we can inspect routing behavior during development.

In [0]:
@mlflow.trace(name="investment_ai_multi_agent_orchestrator")
async def run_multi_agent_orchestrator(
    user_question: str,
    max_iterations: int = 10,
) -> dict:

    messages = [
        {
            "role": "user",
            "content": user_question,
        }
    ]

    tool_history = []

    for _ in range(max_iterations):
        response = await anthropic_client.messages.create(
            model=ORCHESTRATOR_MODEL,
            max_tokens=4096,
            system=ORCHESTRATOR_INSTRUCTIONS,
            messages=messages,
            tools=ORCHESTRATOR_TOOLS,
        )

        text_parts = []
        tool_results = []
        has_tool_use = False

        for block in response.content:
            if block.type == "text":
                text_parts.append(block.text)

            elif block.type == "tool_use":
                has_tool_use = True

                tool_name = block.name
                tool_input = block.input

                if tool_name == "ask_research_agent":
                    specialist_result = await run_research_agent(
                        tool_input["question"]
                    )

                    tool_payload = {
                        "specialist": "Research Agent",
                        "answer": specialist_result,
                    }

                elif tool_name == "ask_portfolio_agent":
                    portfolio_result = ask_portfolio_agent(
                        tool_input["question"]
                    )

                    tool_payload = {
                        "specialist": "Portfolio Agent",
                        "answer": portfolio_result["answer"],
                        "sql": portfolio_result["sql"],
                    }

                else:
                    tool_payload = {
                        "error": f"Unknown tool: {tool_name}"
                    }

                tool_history.append({
                    "tool": tool_name,
                    "input": tool_input,
                })

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(
                        tool_payload,
                        default=str,
                    ),
                })

        if has_tool_use:
            messages.append({
                "role": "assistant",
                "content": response.content,
            })

            messages.append({
                "role": "user",
                "content": tool_results,
            })

        else:
            return {
                "answer": "\n".join(text_parts),
                "tool_history": tool_history,
            }

    return {
        "answer": (
            "The orchestrator reached its maximum tool-call iterations "
            "without producing a final answer."
        ),
        "tool_history": tool_history,
    }


print("✅ Multi-Agent Orchestrator created.")

## 9. Helper to show routing behavior

For the POC, we want to see not only the final answer but also which specialist agents were selected.

In production, this routing information would normally remain part of our trace and observability layer rather than the end-user response.

In [0]:
def display_orchestrator_result(result: dict):
    print("=" * 80)
    print("FINAL ANSWER")
    print("=" * 80)
    print(result["answer"])

    print("\n" + "=" * 80)
    print("SPECIALIST ROUTING")
    print("=" * 80)

    if not result["tool_history"]:
        print("No specialist tool was called.")
        return

    for i, call in enumerate(result["tool_history"], start=1):
        print(f"{i}. {call['tool']}")
        print(f"   Question: {call['input'].get('question')}")

## 10. Routing test — Research only

A research-only question should route to the Research Agent and should not call the Portfolio Agent.

In [0]:
research_question = (
    "Why did Synthetic Compute Company 01 "
    "lower its outlook in Q2 2026?"
)

research_result = await run_multi_agent_orchestrator(
    research_question
)

display_orchestrator_result(research_result)

## 11. Routing test — Portfolio only

A structured portfolio question should route directly to the Genie Portfolio Agent.

The Research Agent is unnecessary for this request.

In [0]:
portfolio_question = (
    "What is our market value exposure by sector "
    "as of June 30, 2026?"
)

portfolio_result = await run_multi_agent_orchestrator(
    portfolio_question
)

display_orchestrator_result(portfolio_result)

## 12. Flagship multi-agent test

Now we ask the business question that motivated the architecture:

> **Which companies lowered their outlook in Q2 2026, why did they lower it, and how much do we own as of June 30, 2026?**

This requires both domains.

Expected reasoning flow:

```text
Question
   ↓
Research Agent
   ↓
Identify companies + reasons + citations
   ↓
Orchestrator receives research result
   ↓
Portfolio Agent
   ↓
Look up ownership / market value for those companies
   ↓
Orchestrator
   ↓
One combined investment-analysis answer
```

In [0]:
flagship_question = (
    "Which companies lowered their outlook in Q2 2026, "
    "why did they lower it, and how much do we own "
    "as of June 30, 2026?"
)

flagship_result = await run_multi_agent_orchestrator(
    flagship_question
)

display_orchestrator_result(flagship_result)

## 13. What we are validating

A successful result is not just a fluent answer.

We want to see that the system made the correct architectural decisions:

```text
Research question
    → Research Agent

Portfolio question
    → Portfolio Agent

Mixed question
    → Research Agent + Portfolio Agent
```

The orchestrator should coordinate specialists rather than gaining direct access to every underlying data source.

That keeps responsibilities clear and makes each capability easier to test, govern, and improve independently.

## 14. Development architecture versus production architecture

This notebook intentionally keeps the orchestration loop visible so we can learn and debug it.

For production, we will not use the legacy managed Supervisor Agent pattern.

The next productionization step is to package the custom orchestrator using **Databricks Agent Bricks**, where Databricks can provide:

- Agent Runtime,
- durable agent serving,
- managed sessions,
- memory when needed,
- governed tool bindings,
- MLflow tracing,
- runtime identity and permissions.

Our specialist responsibilities remain the same; Agent Bricks becomes the production harness around the custom orchestration logic.

# ✅ Notebook 10 complete

We now have a working multi-agent architecture:

```text
                         User
                          ↓
               Multi-Agent Orchestrator
                          ↓
              Claude Sonnet 5.5
                          ↓
           ┌──────────────┴──────────────┐
           │                             │
    Research Agent                Portfolio Agent
           │                             │
      AI Search                    Genie Agent
           │                             │
 Research evidence                Metric View
           └──────────────┬──────────────┘
                          ↓
                 Coordinated answer
```

### Next step — Add governed business logic with Unity Catalog Functions

“In the next notebook, we’ll extend our multi-agent system with a governed Unity Catalog function for deterministic portfolio business logic, such as concentration-risk checks. After that, we’ll evaluate the agent quality and routing behavior before moving into Agent Bricks production packaging.”